# 05. 用户与权限

AGDR VLA 入门教程 · 第 2 章 · 第 1 周

- **学习目标**：读懂 `ls -l` 的权限，会用 `chmod`、`chown`，知道什么时候该用 `sudo`
- **前置知识**：[04-管道与重定向](./04-管道与重定向.ipynb)
- **预计学习时长**：1 小时

## 用户和用户组

Linux 是多用户系统，每个文件都有所有者，每个操作都以某个用户的身份运行。

- **root**：超级用户，不受权限限制，也因此最危险
- **普通用户**：日常使用的身份，一般只能修改自己的文件
- **用户组**：把多个用户编成一组，方便统一授权。例如 `dialout` 组的成员可以访问串口设备

In [ ]:
%%bash
whoami
id

## 读懂 ls -l

In [ ]:
%%bash
rm -rf /tmp/linux-demo && mkdir -p /tmp/linux-demo/dir && cd /tmp/linux-demo
echo 'echo hello from script' > run.sh
ls -l

典型的一行长这样：

```text
-rwxr-xr--  1  shiro  robot  1024  Mar 1 10:00  run.sh
│└┬┘└┬┘└┬┘     └─┬─┘  └─┬─┘  └┬─┘  └────┬───┘  └─┬──┘
│ │  │  │        │      │     │         │        文件名
│ │  │  │        所有者  所属组  大小      修改时间
│ │  │  其他人的权限
│ │  所属组的权限
│ 所有者的权限
类型：- 文件，d 目录，l 符号链接
```

### r、w、x 的含义

| 权限 | 对文件 | 对目录 |
| --- | --- | --- |
| `r` 读 | 查看内容 | 列出目录内容 |
| `w` 写 | 修改内容 | 在目录里新建、删除、重命名文件 |
| `x` 执行 | 作为程序运行 | 进入目录（`cd`） |

所以 `rwxr-xr--` 的意思是：所有者可读写执行，所属组可读和执行，其他人只能读。

## chmod：修改权限

### 数字写法

`r=4`、`w=2`、`x=1`，把每组权限加起来，三个数字依次对应所有者、所属组、其他人：

| 数字 | 权限 | 常见用途 |
| --- | --- | --- |
| `644` | `rw-r--r--` | 普通文件 |
| `755` | `rwxr-xr-x` | 脚本、程序、目录 |
| `600` | `rw-------` | 私密文件，例如 SSH 私钥 |
| `700` | `rwx------` | 私密目录，例如 `~/.ssh` |

### 符号写法

`u` 所有者、`g` 所属组、`o` 其他人、`a` 全部；`+` 添加、`-` 移除。例如 `chmod u+x run.sh`、`chmod go-w a.txt`。

下面的演示说明：脚本要有 `x` 权限才能直接运行。

In [ ]:
%%bash
cd /tmp/linux-demo
chmod 644 run.sh
ls -l run.sh
./run.sh || echo "运行失败，退出码 $?"

In [ ]:
%%bash
cd /tmp/linux-demo
chmod +x run.sh
ls -l run.sh
./run.sh

看到 **Permission denied** 时，先用 `ls -l` 看权限，再判断缺什么。这个报错以后还会经常见到。

目录少了 `x` 权限就进不去，即使有 `r` 权限：

In [ ]:
%%bash
cd /tmp/linux-demo
chmod 600 dir
( cd dir ) || echo "进不去"
chmod 755 dir
( cd dir ) && echo "现在可以进入了"

## sudo：临时使用 root 权限

`sudo <命令>` 用 root 身份运行一条命令，需要输入**你自己的**密码（输入时屏幕上不显示，这是正常的）。

```bash
sudo apt update
```

使用原则：

- **只在需要时用**：修改系统文件、安装软件才需要
- **报错时先看原因**：不要一遇到 Permission denied 就加 `sudo`。用 `sudo` 运行普通程序，可能生成一堆属于 root 的文件，之后你自己反而改不了
- **`sudo` + `rm` 要加倍小心**：没有任何权限能拦住你

> `sudo` 不能写进 notebook 单元格，因为它要等你输入密码。所有带 `sudo` 的命令都请在终端里运行。

## chown：修改所有者

```bash
sudo chown shiro file.txt            # 修改所有者
sudo chown shiro:robot file.txt      # 同时修改所有者和所属组
sudo chown -R shiro:shiro ~/project  # 递归修改整个目录
```

修改所有者通常需要 `sudo`。最常见的用途是修复被 `sudo` 误创建、归 root 所有的文件。

## 加入用户组：机器人开发的常见问题

机械臂常通过串口连接。普通用户直接打开 `/dev/ttyUSB0` 经常会得到 Permission denied。**正确的做法**是把自己加入能访问串口的用户组，而不是把设备权限改成 777：

```bash
ls -l /dev/ttyUSB0                # 查看设备属于哪个组，通常是 dialout
sudo usermod -aG dialout $USER    # 把自己加入 dialout 组
```

`-aG` 里的 `a` 表示追加，**漏掉它会把你从其他组里移除**。设置完要**注销并重新登录**才会生效。

> 没有串口设备的话，现在不用运行，了解即可，第 6、7 章会用到。

## 不要用 chmod 777

`777` 表示所有人都能读、写、执行，遇到权限问题时网上经常有人这么建议。它只是绕过了问题，还带来安全隐患。尤其不要运行 `chmod -R 777`，系统目录被这样改过后可能难以恢复。

遇到权限问题，按这个顺序想：

1. `ls -l`：这个文件属于谁？当前权限是什么？
2. `id`：我是谁？在哪些组里？
3. 只补上缺的那一项：`chmod u+x`、加入用户组，或者 `chown`

## 动手试试

在 `/tmp` 下完成，不计分：

1. 新建 `hello.sh`，内容为 `echo hi`，把权限设为只有你自己可以读写执行
2. 用 `ls -l` 确认权限显示为 `rwx------`，然后用 `./hello.sh` 运行它
3. 说出 `640` 对应的 `rwx` 写法

<details>
<summary>参考答案</summary>

```bash
cd /tmp
echo 'echo hi' > hello.sh
chmod 700 hello.sh
ls -l hello.sh
./hello.sh
```

第 3 题：`rw-r-----`，所有者可读写，所属组只读，其他人没有权限。

</details>

## 小结

| 你应该能 | 相关内容 |
| --- | --- |
| 读懂 `ls -l` 的权限和所有者 | 读懂 ls -l |
| 说出 r、w、x 对文件和目录的不同含义 | r、w、x 的含义 |
| 用数字和符号两种写法运行 `chmod` | chmod |
| 说明 `sudo` 的使用原则 | sudo |
| 用合理方式解决权限问题，不用 777 | 不要用 chmod 777 |

第 1 周到这里结束。下一节：[06-环境变量与Shell脚本](./06-环境变量与Shell脚本.ipynb)